In [1]:
# ============================================================
# NLP + K-MEANS CLUSTERING
# FILE: inconsistent-column-number.csv
# ============================================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------

import csv
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")


# -----------------------------
# 2. FILE NAME
# -----------------------------

FILE_NAME = "inconsistent-column-number.csv"

print("File:", FILE_NAME)


# -----------------------------
# 3. READ CSV
# -----------------------------

with open(
    FILE_NAME,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    reader = csv.reader(file)
    rows = list(reader)

if len(rows) == 0:
    raise ValueError("The CSV file is empty.")

print("Total rows:", len(rows))


# -----------------------------
# 4. FIND MAXIMUM COLUMNS
# -----------------------------

column_counts = [len(row) for row in rows]

max_columns = max(column_counts)

print("Maximum number of columns:", max_columns)


# -----------------------------
# 5. CREATE HEADER
# -----------------------------

header = rows[0].copy()

while len(header) < max_columns:

    header.append(
        f"extra_column_{len(header) + 1}"
    )

header = header[:max_columns]

print("\nColumns:")
print(header)


# -----------------------------
# 6. FIX INCONSISTENT ROWS
# -----------------------------

fixed_rows = []

for row in rows[1:]:

    # Add missing columns
    if len(row) < max_columns:

        row = row + [
            np.nan
        ] * (
            max_columns - len(row)
        )

    # Remove extra columns
    elif len(row) > max_columns:

        row = row[:max_columns]

    fixed_rows.append(row)


# -----------------------------
# 7. CREATE DATAFRAME
# -----------------------------

df = pd.DataFrame(
    fixed_rows,
    columns=header
)

print("\nDataset shape:")
print(df.shape)

display(df.head())


# -----------------------------
# 8. SHOW MISSING VALUES
# -----------------------------

print("\nMissing values:")

print(
    df.isnull().sum()
)


# -----------------------------
# 9. FIND TEXT COLUMN
# -----------------------------

text_scores = {}

for column in df.columns:

    values = (
        df[column]
        .dropna()
        .astype(str)
    )

    if len(values) == 0:

        text_scores[column] = 0

    else:

        text_scores[column] = (
            values.str.len().mean()
        )


print("\nAverage text length:")

for column, score in text_scores.items():

    print(
        column,
        "->",
        round(score, 2)
    )


# Select the column containing the most text

TEXT_COLUMN = max(
    text_scores,
    key=text_scores.get
)

print(
    "\nSelected text column:",
    TEXT_COLUMN
)


# ------------------------------------------------------------
# If you know the exact text column, use this instead:
#
# TEXT_COLUMN = "text"
#
# ------------------------------------------------------------


# -----------------------------
# 10. TEXT CLEANING
# -----------------------------

def clean_text(text):

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove HTML
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Keep letters and spaces
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# -----------------------------
# 11. APPLY NLP CLEANING
# -----------------------------

df["clean_text"] = (
    df[TEXT_COLUMN]
    .fillna("")
    .apply(clean_text)
)

print("\nCleaned text:")

display(
    df[
        [
            TEXT_COLUMN,
            "clean_text"
        ]
    ].head(10)
)


# -----------------------------
# 12. REMOVE EMPTY TEXT
# -----------------------------

original_rows = len(df)

df = df[
    df["clean_text"].str.len() > 0
].copy()

df.reset_index(
    drop=True,
    inplace=True
)

print(
    "\nOriginal rows:",
    original_rows
)

print(
    "Rows after cleaning:",
    len(df)
)


# -----------------------------
# 13. TF-IDF
# -----------------------------

print("\nCreating TF-IDF features...")

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

X = vectorizer.fit_transform(
    df["clean_text"]
)

print(
    "TF-IDF shape:",
    X.shape
)


# -----------------------------
# 14. VOCABULARY
# -----------------------------

features = (
    vectorizer
    .get_feature_names_out()
)

print(
    "\nNumber of words/features:",
    len(features)
)

print(
    "\nFirst 50 features:"
)

print(
    features[:50]
)


# -----------------------------
# 15. CHOOSE K VALUES
# -----------------------------

number_of_documents = X.shape[0]

if number_of_documents < 3:

    raise ValueError(
        "At least 3 text documents are required."
    )

MAX_K = min(
    10,
    number_of_documents - 1
)

K_VALUES = range(
    2,
    MAX_K + 1
)


# -----------------------------
# 16. ELBOW METHOD
# -----------------------------

inertias = []

for k in K_VALUES:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertias.append(
        model.inertia_
    )


plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(K_VALUES),
    inertias,
    marker="o"
)

plt.title(
    "Elbow Method for K-Means"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Inertia"
)

plt.xticks(
    list(K_VALUES)
)

plt.grid()

plt.show()


# -----------------------------
# 17. SILHOUETTE SCORE
# -----------------------------

silhouette_scores = []

for k in K_VALUES:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(
        score
    )

    print(
        f"K = {k}, "
        f"Silhouette Score = {score:.4f}"
    )


# -----------------------------
# 18. SILHOUETTE GRAPH
# -----------------------------

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    list(K_VALUES),
    silhouette_scores,
    marker="o"
)

plt.title(
    "Silhouette Score"
)

plt.xlabel(
    "Number of Clusters (K)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.xticks(
    list(K_VALUES)
)

plt.grid()

plt.show()


# -----------------------------
# 19. SELECT BEST K
# -----------------------------

best_index = np.argmax(
    silhouette_scores
)

BEST_K = list(
    K_VALUES
)[best_index]

print(
    "\nBest K:",
    BEST_K
)

print(
    "Best Silhouette Score:",
    round(
        silhouette_scores[best_index],
        4
    )
)


# -----------------------------
# 20. K-MEANS MODEL
# -----------------------------

kmeans = KMeans(
    n_clusters=BEST_K,
    random_state=42,
    n_init=10
)

df["cluster"] = (
    kmeans.fit_predict(X)
)


print(
    "\nK-Means clustering completed!"
)


# -----------------------------
# 21. CLUSTER COUNTS
# -----------------------------

print(
    "\nDocuments in each cluster:"
)

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(
    cluster_counts
)


# -----------------------------
# 22. TOP WORDS FOR EACH CLUSTER
# -----------------------------

print(
    "\n======================================"
)

print(
    "TOP WORDS IN EACH CLUSTER"
)

print(
    "======================================"
)

cluster_centers = (
    kmeans.cluster_centers_
)

for cluster_number in range(BEST_K):

    top_indices = (
        cluster_centers[
            cluster_number
        ]
        .argsort()[-20:][::-1]
    )

    top_words = [
        features[i]
        for i in top_indices
    ]

    print(
        f"\nCluster {cluster_number}:"
    )

    print(
        ", ".join(top_words)
    )


# -----------------------------
# 23. DISPLAY CLUSTERED DATA
# -----------------------------

print(
    "\n======================================"
)

print(
    "CLUSTERED DATA"
)

print(
    "======================================"
)

display(
    df[
        [
            TEXT_COLUMN,
            "cluster"
        ]
    ].sort_values(
        "cluster"
    ).head(50)
)


# -----------------------------
# 24. 2D VISUALIZATION
# -----------------------------

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2D = svd.fit_transform(
    X
)


# -----------------------------
# 25. PLOT K-MEANS CLUSTERS
# -----------------------------

plt.figure(
    figsize=(12, 8)
)

scatter = plt.scatter(
    X_2D[:, 0],
    X_2D[:, 1],
    c=df["cluster"],
    alpha=0.7
)

plt.xlabel(
    "SVD Component 1"
)

plt.ylabel(
    "SVD Component 2"
)

plt.title(
    "K-Means NLP Clusters"
)

plt.colorbar(
    scatter,
    label="Cluster"
)

plt.grid()

plt.show()


# -----------------------------
# 26. CLUSTER SIZE GRAPH
# -----------------------------

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Number of Documents"
)

plt.title(
    "Documents per Cluster"
)

plt.grid(
    axis="y"
)

plt.show()


# -----------------------------
# 27. SAMPLE DOCUMENTS
# -----------------------------

print(
    "\n======================================"
)

print(
    "SAMPLE DOCUMENTS BY CLUSTER"
)

print(
    "======================================"
)

for cluster_number in range(BEST_K):

    print(
        f"\n---------- CLUSTER {cluster_number} ----------"
    )

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    samples = cluster_data[
        [TEXT_COLUMN]
    ].head(5)

    for i, row in samples.iterrows():

        print(
            "\nDocument:",
            i
        )

        print(
            str(
                row[TEXT_COLUMN]
            )[:500]
        )


# -----------------------------
# 28. FINAL SILHOUETTE SCORE
# -----------------------------

final_score = silhouette_score(
    X,
    df["cluster"]
)

print(
    "\n======================================"
)

print(
    "FINAL RESULTS"
)

print(
    "======================================"
)

print(
    "File:",
    FILE_NAME
)

print(
    "Text column:",
    TEXT_COLUMN
)

print(
    "Number of documents:",
    len(df)
)

print(
    "Number of TF-IDF features:",
    X.shape[1]
)

print(
    "Number of clusters:",
    BEST_K
)

print(
    "Silhouette Score:",
    round(
        final_score,
        4
    )
)


# -----------------------------
# 29. SAVE FINAL DATASET
# -----------------------------

OUTPUT_FILE = (
    "inconsistent-column-number_clustered.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "\nOutput file created:"
)

print(
    OUTPUT_FILE
)


# -----------------------------
# 30. SAVE CLUSTER SUMMARY
# -----------------------------

SUMMARY_FILE = (
    "inconsistent-column-number_cluster_summary.csv"
)

summary = pd.DataFrame({
    "Cluster": cluster_counts.index,
    "Number_of_Documents": cluster_counts.values
})

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    "Summary file created:"
)

print(
    SUMMARY_FILE
)


# -----------------------------
# 31. DONE
# -----------------------------

print(
    "\n======================================"
)

print(
    "NLP + K-MEANS ANALYSIS COMPLETED"
)

print(
    "======================================"
)

File: inconsistent-column-number.csv
Total rows: 3
Maximum number of columns: 3

Columns:
['number', 'string', 'boolean']

Dataset shape:
(2, 3)


,number,string,boolean
0,1,one,true
1,2,two,NaN



Missing values:
number     0
string     0
boolean    1
dtype: int64

Average text length:
number -> 1.0
string -> 3.0
boolean -> 4.0

Selected text column: boolean

Cleaned text:


,boolean,clean_text
0,true,true
1,NaN,



Original rows: 2
Rows after cleaning: 1

Creating TF-IDF features...


ValueError: max_df corresponds to < documents than min_df